In [1]:
import tensorflow as tf
from tensorflow.keras import layers, Model, Input

In [2]:
# =========================
# IBRBlock(CNN)
# =========================
@tf.keras.utils.register_keras_serializable(package="IBRSwin")
class IBRBlock(layers.Layer):
    def __init__(self, filters, stride=1, expansion=4, se_ratio=0.25, dropout=0.0, **kwargs):
        super().__init__(**kwargs)
        self.filters = filters
        self.stride = stride
        self.expansion = expansion
        self.se_ratio = se_ratio
        self.dropout = dropout

        self.expand_conv = None
        self.expand_bn = None
        self.dw_conv = None
        self.dw_bn = None
        self.project_conv = None
        self.project_bn = None
        self.drop = layers.Dropout(dropout)

        self.shortcut_proj = None
        self.shortcut_bn = None

        self.act = layers.Activation("swish")

    def build(self, input_shape):
        in_channels = int(input_shape[-1])
        hidden_dim = in_channels * self.expansion

        self.expand_conv = layers.Conv2D(hidden_dim, 1, padding="same", use_bias=False)
        self.expand_bn = layers.BatchNormalization()

        self.dw_conv = layers.DepthwiseConv2D(
            3, strides=self.stride, padding="same", use_bias=False
        )
        self.dw_bn = layers.BatchNormalization()

        se_hidden = max(8, int(hidden_dim * self.se_ratio))
        self.se_gap = layers.GlobalAveragePooling2D()
        self.se_fc1 = layers.Dense(se_hidden, activation="swish")
        self.se_fc2 = layers.Dense(hidden_dim, activation="sigmoid")

        self.project_conv = layers.Conv2D(self.filters, 1, padding="same", use_bias=False)
        self.project_bn = layers.BatchNormalization()

        if self.stride != 1 or in_channels != self.filters:
            self.shortcut_proj = layers.Conv2D(self.filters, 1, strides=self.stride, padding="same", use_bias=False)
            self.shortcut_bn = layers.BatchNormalization()

        super().build(input_shape)

    def call(self, x, training=None):
        shortcut = x

        x = self.expand_conv(x)
        x = self.expand_bn(x, training=training)
        x = self.act(x)

        x = self.dw_conv(x)
        x = self.dw_bn(x, training=training)
        x = self.act(x)

        se = self.se_gap(x)
        se = self.se_fc1(se)
        se = self.se_fc2(se)
        se = tf.reshape(se, [-1, 1, 1, tf.shape(x)[-1]])
        x = x * se

        x = self.project_conv(x)
        x = self.project_bn(x, training=training)

        if self.shortcut_proj is not None:
            shortcut = self.shortcut_proj(shortcut)
            shortcut = self.shortcut_bn(shortcut, training=training)

        x = layers.Add()([x, shortcut])
        x = self.act(x)
        x = self.drop(x, training=training)
        return x



In [3]:
# =========================
# Swin Block
# =========================
@tf.keras.utils.register_keras_serializable(package="IBRSwin")
class SwinBlock(layers.Layer):
    def __init__(self, dim, num_heads=8, window_size=7, shift_size=0, mlp_ratio=4, dropout=0.0, **kwargs):
        super().__init__(**kwargs)

        if dim % num_heads != 0:
            raise ValueError(f"dim={dim} must be divisible by num_heads={num_heads}")

        self.dim = dim
        self.num_heads = num_heads
        self.window_size = window_size
        self.shift_size = shift_size
        self.mlp_ratio = mlp_ratio
        self.dropout = dropout

        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.qkv = layers.Dense(dim * 3, use_bias=False)
        self.proj = layers.Dense(dim)
        self.attn_drop = layers.Dropout(dropout)
        self.proj_drop = layers.Dropout(dropout)

        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.mlp = tf.keras.Sequential([
            layers.Dense(dim * mlp_ratio, activation="gelu"),
            layers.Dropout(dropout),
            layers.Dense(dim),
        ])

    def build(self, input_shape):
        self.relative_position_bias_table = self.add_weight(
            shape=((2 * self.window_size - 1) * (2 * self.window_size - 1), self.num_heads),
            initializer="zeros",
            trainable=True,
            name="rel_pos_bias"
        )

        coords_h = tf.range(self.window_size)
        coords_w = tf.range(self.window_size)
        coords = tf.stack(tf.meshgrid(coords_h, coords_w, indexing="ij"))
        coords_flatten = tf.reshape(coords, (2, -1))

        relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]
        relative_coords = tf.transpose(relative_coords, [1, 2, 0])

        relative_coords = relative_coords + self.window_size - 1
        relative_coords = relative_coords[:, :, 0] * (2 * self.window_size - 1) + relative_coords[:, :, 1]

        self.relative_position_index = tf.Variable(
            initial_value=tf.cast(relative_coords, tf.int32),
            trainable=False,
            name="rel_pos_index"
        )

        super().build(input_shape)

    def window_partition(self, x):
        B = tf.shape(x)[0]
        H = tf.shape(x)[1]
        W = tf.shape(x)[2]
        C = tf.shape(x)[3]
        ws = self.window_size

        pad_h = (ws - H % ws) % ws
        pad_w = (ws - W % ws) % ws

        x = tf.pad(x, [[0, 0], [0, pad_h], [0, pad_w], [0, 0]])

        Hp = tf.shape(x)[1]
        Wp = tf.shape(x)[2]

        x = tf.reshape(x, [B, Hp // ws, ws, Wp // ws, ws, C])
        x = tf.transpose(x, [0, 1, 3, 2, 4, 5])
        windows = tf.reshape(x, [-1, ws * ws, C])

        return windows, B, Hp, Wp

    def window_reverse(self, windows, B, Hp, Wp):
        ws = self.window_size
        C = tf.shape(windows)[-1]

        x = tf.reshape(windows, [B, Hp // ws, Wp // ws, ws, ws, C])
        x = tf.transpose(x, [0, 1, 3, 2, 4, 5])
        x = tf.reshape(x, [B, Hp, Wp, C])
        return x

    def call(self, inputs, training=None):
        shortcut = inputs
        x = self.norm1(inputs)

        if self.shift_size > 0:
            x = tf.roll(x, shift=[-self.shift_size, -self.shift_size], axis=[1, 2])

        windows, B, Hp, Wp = self.window_partition(x)

        qkv = self.qkv(windows)
        qkv = tf.reshape(qkv, [-1, tf.shape(windows)[1], 3, self.num_heads, self.dim // self.num_heads])
        qkv = tf.transpose(qkv, [2, 0, 3, 1, 4])

        q, k, v = qkv[0], qkv[1], qkv[2]

        scale = (self.dim // self.num_heads) ** -0.5
        attn = tf.matmul(q, k, transpose_b=True) * scale

        bias = tf.gather(self.relative_position_bias_table,
                         tf.reshape(self.relative_position_index, [-1]))
        bias = tf.reshape(bias, [self.window_size * self.window_size,
                                 self.window_size * self.window_size,
                                 self.num_heads])
        bias = tf.transpose(bias, [2, 0, 1])

        attn = attn + tf.expand_dims(bias, 0)
        attn = tf.nn.softmax(attn, axis=-1)
        attn = self.attn_drop(attn, training=training)

        out = tf.matmul(attn, v)
        out = tf.transpose(out, [0, 2, 1, 3])
        out = tf.reshape(out, [-1, self.window_size * self.window_size, self.dim])

        out = self.proj(out)
        out = self.proj_drop(out, training=training)

        x = self.window_reverse(out, B, Hp, Wp)

        if self.shift_size > 0:
            x = tf.roll(x, shift=[self.shift_size, self.shift_size], axis=[1, 2])

        x = x[:, :tf.shape(shortcut)[1], :tf.shape(shortcut)[2], :]
        x = shortcut + x
        x = x + self.mlp(self.norm2(x), training=training)

        return x



In [4]:



# =========================
# Patch Embedding
# =========================
@tf.keras.utils.register_keras_serializable(package="IBRSwin")
class PatchEmbed(layers.Layer):
    def __init__(self, embed_dim=192, patch_size=4, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.patch_size = patch_size

        self.proj = layers.Conv2D(embed_dim, kernel_size=patch_size, strides=patch_size, padding="same", use_bias=False)
        self.bn = layers.BatchNormalization()
        self.act = layers.Activation("swish")
        self.norm = layers.LayerNormalization(epsilon=1e-6)

    def call(self, x, training=None):
        x = self.proj(x)
        x = self.bn(x, training=training)
        x = self.act(x)
        x = self.norm(x)
        return x


# =========================
# Patch Merging
# =========================
@tf.keras.utils.register_keras_serializable(package="IBRSwin")
class PatchMerging(layers.Layer):
    def __init__(self, out_dim, **kwargs):
        super().__init__(**kwargs)
        self.out_dim = out_dim
        self.norm = layers.LayerNormalization(epsilon=1e-6)
        self.reduction = None

    def build(self, input_shape):
        self.reduction = layers.Dense(self.out_dim, use_bias=False)
        super().build(input_shape)

    def call(self, x):
        # x: (B, H, W, C)
        B = tf.shape(x)[0]
        H = tf.shape(x)[1]
        W = tf.shape(x)[2]
        C = tf.shape(x)[3]

        pad_h = H % 2
        pad_w = W % 2
        x = tf.pad(x, [[0, 0], [0, pad_h], [0, pad_w], [0, 0]])

        H2 = tf.shape(x)[1]
        W2 = tf.shape(x)[2]

        x0 = x[:, 0:H2:2, 0:W2:2, :]
        x1 = x[:, 1:H2:2, 0:W2:2, :]
        x2 = x[:, 0:H2:2, 1:W2:2, :]
        x3 = x[:, 1:H2:2, 1:W2:2, :]

        x = tf.concat([x0, x1, x2, x3], axis=-1)
        x = self.norm(x)
        x = self.reduction(x)
        return x



In [5]:
# =========================
# Swin Stage
# =========================
def swin_stage(x, dim, depth, num_heads, window_size=7, dropout=0.0, name_prefix="swin"):
    for i in range(depth):
        shift = 0 if i % 2 == 0 else window_size // 2
        x = SwinBlock(
            dim=dim,
            num_heads=num_heads,
            window_size=window_size,
            shift_size=shift,
            mlp_ratio=4,
            dropout=dropout,
            name=f"{name_prefix}_block_{i+1}"
        )(x)
    return x



In [6]:
# =========================
# Bi-Directional Cross Attention Fusion
# =========================
@tf.keras.utils.register_keras_serializable(package="IBRSwin")
class BiDirectionalCrossAttentionFusion(layers.Layer):
    def __init__(self, fusion_dim, num_heads=8, mlp_ratio=4, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.fusion_dim = fusion_dim
        self.num_heads = num_heads
        self.mlp_ratio = mlp_ratio
        self.dropout = dropout

        self.cnn_proj = layers.Conv2D(fusion_dim, 1, padding="same", use_bias=False)
        self.swin_proj = layers.Conv2D(fusion_dim, 1, padding="same", use_bias=False)

        self.cnn_bn = layers.BatchNormalization()
        self.swin_bn = layers.BatchNormalization()

        self.norm_c = layers.LayerNormalization(epsilon=1e-6)
        self.norm_s = layers.LayerNormalization(epsilon=1e-6)
        self.norm_f = layers.LayerNormalization(epsilon=1e-6)

        self.attn_c_to_s = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=fusion_dim // num_heads, dropout=dropout, name="attn_c_to_s"
        )
        self.attn_s_to_c = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=fusion_dim // num_heads, dropout=dropout, name="attn_s_to_c"
        )

        self.drop = layers.Dropout(dropout)

        self.ffn = tf.keras.Sequential([
            layers.Dense(fusion_dim * mlp_ratio, activation="gelu"),
            layers.Dropout(dropout),
            layers.Dense(fusion_dim),
        ])

        self.gate = layers.Dense(fusion_dim, activation="sigmoid")

    def _resize_if_needed(self, x, target_hw):
        th, tw = target_hw
        xh = tf.shape(x)[1]
        xw = tf.shape(x)[2]
        return tf.cond(
            tf.logical_or(tf.not_equal(xh, th), tf.not_equal(xw, tw)),
            lambda: tf.image.resize(x, (th, tw), method="bilinear"),
            lambda: x
        )

    def call(self, inputs, training=None):
        cnn_feat, swin_feat = inputs  # (B,H,W,C1), (B,H,W,C2)

        target_hw = (tf.shape(cnn_feat)[1], tf.shape(cnn_feat)[2])
        swin_feat = self._resize_if_needed(swin_feat, target_hw)

        cnn_feat = self.cnn_proj(cnn_feat)
        cnn_feat = self.cnn_bn(cnn_feat, training=training)

        swin_feat = self.swin_proj(swin_feat)
        swin_feat = self.swin_bn(swin_feat, training=training)

        B = tf.shape(cnn_feat)[0]
        H = tf.shape(cnn_feat)[1]
        W = tf.shape(cnn_feat)[2]

        cnn_tokens = tf.reshape(cnn_feat, [B, H * W, self.fusion_dim])
        swin_tokens = tf.reshape(swin_feat, [B, H * W, self.fusion_dim])

        cnn_tokens_n = self.norm_c(cnn_tokens)
        swin_tokens_n = self.norm_s(swin_tokens)

        # Bi-directional cross attention
        cnn_to_swin = self.attn_c_to_s(query=cnn_tokens_n, value=swin_tokens_n, key=swin_tokens_n, training=training)
        swin_to_cnn = self.attn_s_to_c(query=swin_tokens_n, value=cnn_tokens_n, key=cnn_tokens_n, training=training)

        cnn_tokens = cnn_tokens + self.drop(cnn_to_swin, training=training)
        swin_tokens = swin_tokens + self.drop(swin_to_cnn, training=training)

        # Fuse both directions
        fused = tf.concat([cnn_tokens, swin_tokens], axis=-1)
        fused = layers.Dense(self.fusion_dim)(fused)
        fused = fused + self.ffn(self.norm_f(fused), training=training)

        # Gated refinement
        gate = self.gate(fused)
        fused = fused * gate + fused

        fused = tf.reshape(fused, [B, H, W, self.fusion_dim])
        return fused



In [9]:



# =========================
# Full Model
# =========================
def build_model(
    input_shape=(224, 224, 3),
    num_classes=7,
    stem_channels=64,
    swin_embed_dim=192,
    swin_heads=8,
    fusion_heads=8,
    expansion=4,
    dropout_rate=0.3,
    window_size=7
):
    inputs = Input(shape=input_shape, name="input_image")

    # -----------------
    # CNN branch
    # -----------------
    x = layers.Conv2D(stem_channels, 7, strides=2, padding="same", use_bias=False, name="stem_conv")(inputs)
    x = layers.BatchNormalization(name="stem_bn")(x)
    x = layers.Activation("swish", name="stem_swish")(x)

    f1 = IBRBlock(64, stride=1, expansion=expansion, dropout=0.0, name="ibr_block_1")(x)
    f2 = IBRBlock(128, stride=2, expansion=expansion, dropout=0.0, name="ibr_block_2")(f1)
    f3 = IBRBlock(256, stride=2, expansion=expansion, dropout=0.0, name="ibr_block_3")(f2)
    f4 = IBRBlock(512, stride=2, expansion=expansion, dropout=0.0, name="ibr_block_4")(f3)

    # -----------------
    # Swin branch
    # -----------------
    s = PatchEmbed(embed_dim=swin_embed_dim, name="patch_embed")(inputs)
    s1 = swin_stage(s, dim=swin_embed_dim, depth=2, num_heads=swin_heads, window_size=window_size, dropout=0.0, name_prefix="swin_stage1")
    s = PatchMerging(out_dim=swin_embed_dim * 2, name="patch_merge_1")(s1)
    s2 = swin_stage(s, dim=swin_embed_dim * 2, depth=2, num_heads=swin_heads, window_size=window_size, dropout=0.0, name_prefix="swin_stage2")
    s = PatchMerging(out_dim=swin_embed_dim * 4, name="patch_merge_2")(s2)
    s3 = swin_stage(s, dim=swin_embed_dim * 4, depth=2, num_heads=swin_heads, window_size=window_size, dropout=0.0, name_prefix="swin_stage3")

    # -----------------
    # Bi-directional fusion stages
    # -----------------
    o1 = BiDirectionalCrossAttentionFusion(
        fusion_dim=192, num_heads=fusion_heads, mlp_ratio=4, dropout=0.1, name="fusion_stage_1"
    )([f2, s1])

    o2 = BiDirectionalCrossAttentionFusion(
        fusion_dim=384, num_heads=fusion_heads, mlp_ratio=4, dropout=0.1, name="fusion_stage_2"
    )([f3, s2])

    o3 = BiDirectionalCrossAttentionFusion(
        fusion_dim=512, num_heads=fusion_heads, mlp_ratio=4, dropout=0.1, name="fusion_stage_3"
    )([f4, s3])

    # -----------------
    # Pool + Head
    # -----------------
    gap_o1 = layers.GlobalAveragePooling2D(name="gap_o1")(o1)
    gap_o2 = layers.GlobalAveragePooling2D(name="gap_o2")(o2)
    gap_o3 = layers.GlobalAveragePooling2D(name="gap_o3")(o3)
    gap_f4 = layers.GlobalAveragePooling2D(name="gap_f4")(f4)
    concat = layers.Concatenate(name="concat_features")([gap_o1, gap_o2, gap_o3, gap_f4])

    x = layers.Dense(512, activation="swish", name="dense_512")(concat)
    x = layers.BatchNormalization(name="head_bn_1")(x)
    x = layers.Dropout(dropout_rate, name="head_dropout_1")(x)

    x = layers.Dense(256, activation="swish", name="dense_256")(x)
    x = layers.BatchNormalization(name="head_bn_2")(x)
    x = layers.Dropout(dropout_rate, name="head_dropout_2")(x)

    outputs = layers.Dense(num_classes, activation="softmax", dtype="float32", name="predictions")(x)

    return Model(inputs=inputs, outputs=outputs, name="fused_IBR4Net_Swin_T")




In [10]:
# =========================
# Build, Summary, Compile
# =========================
tf.keras.backend.clear_session()

model_224 = build_model(
    input_shape=(224, 224, 3),
    num_classes=7,
    stem_channels=64,
    swin_embed_dim=192,
    swin_heads=8,
    fusion_heads=8,
    expansion=4,
    dropout_rate=0.3,
    window_size=7
)

model_224.summary()

Model: "fused_IBR4Net_Swin_T"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_image         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ patch_embed         │ (None, 56, 56,    │     10,368 │ input_image[0][0] │
│ (PatchEmbed)        │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ swin_stage1_block_1 │ (None, 56, 56,    │    445,640 │ patch_embed[0][0] │
│ (SwinBlock)         │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │      9,408 │ input_image[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ swin_stage1_block_2 │ (None, 56, 56,    │    445,640 │ swin_stage1_bloc… │
│ (SwinBlock)         │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        256 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ patch_merge_1       │ (None, 28, 28,    │    296,448 │ swin_stage1_bloc… │
│ (PatchMerging)      │ 384)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_swish          │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ swin_stage2_block_1 │ (None, 28, 28,    │  1,774,664 │ patch_merge_1[0]… │
│ (SwinBlock)         │ 384)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ibr_block_1         │ (None, 112, 112,  │     70,464 │ stem_swish[0][0]  │
│ (IBRBlock)          │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ swin_stage2_block_2 │ (None, 28, 28,    │  1,774,664 │ swin_stage2_bloc… │
│ (SwinBlock)         │ 384)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ibr_block_2         │ (None, 56, 56,    │     95,808 │ ibr_block_1[0][0] │
│ (IBRBlock)          │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ patch_merge_2       │ (None, 14, 14,    │  1,182,720 │ swin_stage2_bloc… │
│ (PatchMerging)      │ 768)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ibr_block_3         │ (None, 28, 28,    │    371,840 │ ibr_block_2[0][0] │
│ (IBRBlock)          │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ swin_stage3_block_1 │ (None, 14, 14,    │  7,086,920 │ patch_merge_2[0]… │
│ (SwinBlock)         │ 768)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ibr_block_4         │ (None, 14, 14,    │  1,464,576 │ ibr_block_3[0][0] │
│ (IBRBlock)          │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ swin_stage3_block_2 │ (None, 14, 14,    │  7,086,920 │ swin_stage3_bloc

 Total params: 31,655,159 (120.75 MB)

 Trainable params: 31,636,855 (120.69 MB)

 Non-trainable params: 18,304 (71.50 KB)